In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
import minsearch
from openai import OpenAI

# 1. Fit the custom index
index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

# 2. Search parameters
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query=query, num_results=5)

# 3. Build a simple context block mapping our schema fields
context_block = ""
for doc in search_results:
    context_block += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"

prompt = f"""
Your task is to answer questions from the course participants based on the provided context.
If the answer is not found in the context, respond with "I don't know."
Question: {query}
Context:
{context_block}
""".strip()

# 4. Route the call to count input tokens directly on the usage metadata
openai_client = OpenAI()
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=[{"role": "user", "content": prompt}]
)

print(f"Input tokens sent: {response.usage.input_tokens}")

Input tokens sent: 7116


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [7]:
# 1. Fit new chunk index
chunk_index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

# 2. Search and assemble prompt
chunk_results = chunk_index.search(query=query, num_results=5)

chunk_context = ""
for chunk in chunk_results:
    chunk_context += f"Filename: {chunk['filename']}\nContent: {chunk['content']}\n\n"

chunk_prompt = f"""
Your task is to answer questions from the course participants based on the provided context.
If the answer is not found in the context, respond with "I don't know."
Question: {query}
Context:
{chunk_context}
""".strip()

# 3. Call model to check token optimization
chunk_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=[{"role": "user", "content": chunk_prompt}]
)

print(f"Chunked input tokens: {chunk_response.usage.input_tokens}")

Chunked input tokens: 2299


In [8]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# 1. Define the framework tool wrapper tracking type hints and docstring descriptions
def search_chunks(query: str) -> list:
    """
    Search the course lesson markdown chunks database for information matching the query terms.
    """
    # Queries the chunk index from Q5
    return chunk_index.search(query=query, num_results=5)

# 2. Add structural schema reflection definitions
agent_tools = Tools()
agent_tools.add_tool(search_chunks)

# 3. Initialize execution manager with explicit nudging helper guidelines
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=IPythonChatInterface(),
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

# 4. Trigger the multi-turn autonomous loop
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?"
)